# Выбор локации для скважины

Допустим, вы работаете в добывающей компании «ГлавРосГосНефть». Нужно решить, где бурить новую скважину.

Вам предоставлены пробы нефти в трёх регионах: в каждом 100 000 месторождений, где измерили качество нефти и объём её запасов. Постройте модель машинного обучения, которая поможет определить регион, где добыча принесёт наибольшую прибыль. Проанализируйте возможную прибыль и риски техникой *Bootstrap.*

Шаги для выбора локации:

- В избранном регионе ищут месторождения, для каждого определяют значения признаков;
- Строят модель и оценивают объём запасов;
- Выбирают месторождения с самым высокими оценками значений. Количество месторождений зависит от бюджета компании и стоимости разработки одной скважины;
- Прибыль равна суммарной прибыли отобранных месторождений.

## Настройка окружения

### Импорты

In [ ]:
# %pip install -Uq scikit-learn
%pip install phik -q
# %pip install shap -q

# imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
# import shap

import warnings
warnings.filterwarnings('ignore', category=pd.core.common.SettingWithCopyWarning)

from phik import phik_matrix
from plotly.subplots import make_subplots
from scipy import stats as st
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error,
#     root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    make_scorer,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PolynomialFeatures,
)

from sklearn.model_selection import (
    train_test_split, 
    GridSearchCV, 
    RandomizedSearchCV,
)

# загружаем класс для работы с пропусками
from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


### Настройки отображения

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [ ]:
# output settings
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_colwidth', None)

### Объявление функций

In [ ]:
def check_size(current_df, original_df):
    print("Количество записей: {}".format(len(current_df)))
    print("Процент от начального объема данных: {:.2%}".format(len(current_df) / len(original_df)))

In [ ]:
import os

HOST = "https://code.s3.yandex.net"

# load csv
def load_csv(dataset_path, **kwargs):
    # check local relative path --> absolute path --> server request
    local_relative_path = "." + dataset_path
    path = local_relative_path if os.path.exists(local_relative_path) else dataset_path if os.path.exists(dataset_path) else HOST + dataset_path
    print("Dataset path:", path)
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

In [ ]:
def df_init_analysis(df_name, df):
    print("-" * 50)
    print(f"{df_name}\n")
    print("Общая информация\n")
    print(df.info())

    print("\nБазовая статистика по данным")
    display(df.describe(include='all'))

    print("\nИнформация по колонкам\n")
    df_init_analysis_column_names = [
        "column_name",
        "type",
        "na_count",
        "empty_count",
        "unique_count",
    ]

    df_init_analysis_data = []
    for column_name in df.columns.tolist():
        df_init_column_data = []
        df_init_column_data.append(column_name)
        df_init_column_data.append(df[column_name].dtype)
        df_init_column_data.append(df[column_name].isna().sum())
        df_init_column_data.append(sum(df[column_name] == ""))
        df_init_column_data.append(df[column_name].nunique())
        df_init_analysis_data.append(df_init_column_data)

    df_init = pd.DataFrame(columns=df_init_analysis_column_names, data=df_init_analysis_data)
    display(df_init)

    print(f"\nКоличество дубликатов: {df.duplicated().sum()}\n")

## Загрузка и подготовка данных

### Загрузка данных

In [ ]:
geo_data_0_original = load_csv("/datasets/geo_data_0.csv")
# geo_data_0_original = load_csv("/datasets/geo_data_0.csv", index_col="id")
geo_data_0 = geo_data_0_original.copy()
geo_data_0_original.head()

In [ ]:
geo_data_1_original = load_csv("/datasets/geo_data_1.csv")
# geo_data_1_original = load_csv("/datasets/geo_data_1.csv", index_col="id")
geo_data_1 = geo_data_1_original.copy()
geo_data_1.head()

In [ ]:
geo_data_2_original = load_csv("/datasets/geo_data_2.csv")
# geo_data_2_original = load_csv("/datasets/geo_data_2.csv", index_col="id")
geo_data_2 = geo_data_2_original.copy()
geo_data_2.head()

### Первичный анализ данных

In [ ]:
df_dict = {
    "geo_data_0": geo_data_0,
    "geo_data_1": geo_data_1,
    "geo_data_2": geo_data_2,
}

In [ ]:
for df_name, df in df_dict.items():
    df_init_analysis(df_name, df)

Несмотря на то, что поле `id` должно быть уникальным, оно таковым не является. В данных присутствуют записи с одинаковым id.

В таблице `geo_data_1` в поле `product` всего 12 уникальных значений. Это выглядит очень странно. Скорее всего, ошибки в данных.

Пустых значений в таблицах нет.

### Подготовка данных

#### Дубликаты id

Найдем все записи с дублями в поле `id`.

In [ ]:
for df_name, df in df_dict.items():
    print("-" * 50)
    print(f"\n{df_name} id duplicates\n")
    id_list = df[df["id"].duplicated()]["id"].to_list()
    display(df.query("id in @id_list").sort_values(by="id"))
    print()

Данные выглядят совсем непохожими, но это уникальные идентификаторы скважин. Для обучения моделей они не будут показательными, а вот сами характеристики нужны.

Оставим записи в таблицах, но заменим индексы на `id`

In [ ]:
for df in df_dict.values():
    df.set_index("id", inplace=True)
    display(df.head())

#### Распределение данных

Построим гистограммы распределения для каждого признака в каждой таблице

In [ ]:
# количество корзин в зависимости от количества уникальных значений
def get_bins_count(value):
    print(f"n_uniq_values: {value}")
    n_bins = value // 100 if value > 1000 else value // 10 if value > 100 else value
    print(f"n_bins: {n_bins}")
    return n_bins

In [ ]:
# histogram function
def get_histogram(df, df_name, df_column_name):
    fig = px.histogram(
        df, 
        x=df_column_name, 
        nbins=get_bins_count(df[df_column_name].nunique()),
        marginal = 'box',
        barmode="group",
        opacity = 0.5,
        title = f'Распределение значений {df_name}.{df_column_name}',
    )

    fig.update_layout(
        # title_text=f"Распределение значений {df_name}.{df_column_name}",
        xaxis_title_text=f"{df_name}.{df_column_name}",
        yaxis_title_text="Кол-во",
    )

    fig.show()

In [ ]:
for df_name, df in df_dict.items():
    for column_name in df.columns.to_list():
        get_histogram(df, df_name, column_name)

Для колонок `f2` и `product` в таблице `geo_data_1` данные выглядят резко выделяющимися на фоне двух других датасетов и больше похожи на категориальные признаки. Построим для них столбчатые и круговые диаграммы.

In [ ]:
# Создание категорий с помощью pd.cut
geo_data_1_cat = geo_data_1.copy()
geo_data_1_cat["f2_cat"] = pd.cut(
    geo_data_1_cat["f2"], 
    bins=[-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], 
    labels=[0, 1, 2, 3, 4, 5]
)
geo_data_1_cat.sample(10)


In [ ]:
# функция по отрисовке столбчатой и круговой диаграммы
def show_bar_pie(df, df_cat_column_name):
    count_column = df.columns.to_list()[0]
    # Сгруппируем данные по признаку
    df = df.groupby(by=[df_cat_column_name])[count_column].count().reset_index().set_axis([df_cat_column_name, "count"], axis=1)
    display(df)
    print()

    # Столбчатая диаграмма
    fig = px.bar(
        df,
        x=df[df_cat_column_name],
        y=df["count"],
    )
    fig.show()
    print()

    # Круговая диаграмма
    fig = px.pie(
        df,
        names=df[df_cat_column_name],
        values=df["count"],
    )
    fig.show()
    print()

In [ ]:
for cat_column_name in ["f2_cat", "product"]:
    print("-" * 50)
    print(f"Диаграммы {cat_column_name}")
    show_bar_pie(geo_data_1_cat, cat_column_name)

Данные по категориям распределены равномерно

#### Объединение датасетов в один

Для объединения будем использовать метод concat, т.к. некоторые индексы повторяются в разных датасетах

In [ ]:
geo_data_full = pd.concat([geo_data_0, geo_data_1, geo_data_2])
print(geo_data_full.shape)
geo_data_full.head()

Добавим в словарь датасетов

In [ ]:
df_dict["geo_data_full"] = geo_data_full
df_dict.keys()

#### Scatter plots

In [ ]:
# список признаков без учета целевого признака
cols_list = geo_data_0.drop(columns=["product"]).columns.to_list()
# количество признаков без учета целевого == количество подграфиков
n_cols = len(cols_list)
n_cols

In [ ]:
for df_name, df in df_dict.items():
    print("\n", "-" * 50)
    print(f"\n{df_name}\n")

    # Создаем фигуру с подграфиками
    fig = make_subplots(rows=1, cols=n_cols)

    # возьмем 10К наблюдений (чтобы не перегружать машину)
    df_cut = df.sample(10000)
    for index, column_name in enumerate(cols_list):
        # Добавляем первый подграфик
        fig.add_trace(
            go.Scatter(
                x=df_cut[column_name],
                y=df_cut["product"],
                mode='markers',
                name=column_name,
            ),
            row=1,
            col=index + 1,
        )

    fig.update_layout(
        title_text=f"{df_name}.product",
    )
    fig.show()

Есть явновыраженные линейные зависимости целевого признака `product` от признака `f2` во всех датасетах. От признаков `f0` и `f1` зависимости нелинейные

#### Корреляция признаков

Построим матрицу корреляции и тепловую карту

In [ ]:
# Создаем фигуру с тремя подграфиками
plt.figure(figsize=(15, 3))

for index, key in enumerate(df_dict):
    # подграфик для каждого датасета
    plt.subplot(1, len(df_dict), index + 1)
    sns.heatmap(df_dict[key].phik_matrix(verbose=False), annot=True, fmt=".2f")
    plt.title(key)

plt.tight_layout()
plt.show()


- `geo_data_0`:
    - сильная корреляция между признаками f0, f1
    - заметная корреляция между признаками f2, product
- `geo_data_1`:
    - мультиколлинеарность между признаками product, f2
    - выраженная корреляция между признаками product, f0
- `geo_data_2`:
    - заметная корреляция между признаками f2, product
- `geo_data_full`:
    - заметная корреляция между признаками f2, product

В качестве возможных решений можно предложить следующие действия для признаков с сильной корреляцией и мультиколлиенарностью:
- Удаление коррелирующих признаков
- Создание новых, композитных переменных
- Обучать модель на объединенном датасете

Воспользуемся вариантом "Обучать модель на объединенном датасете"

#### Создание новых признаков

##### geo_data_0

In [ ]:
geo_data_0_prepared = geo_data_0.copy()
geo_data_0_prepared["f0_f1_prod"] = geo_data_0_prepared["f0"] * geo_data_0_prepared["f1"]
geo_data_0_prepared = geo_data_0_prepared.drop(["f0", "f1"], axis=1)
geo_data_0_prepared.head()

##### geo_data_1

In [ ]:
geo_data_1_prepared = geo_data_1.copy()
geo_data_1_prepared["f0_f2_prod"] = geo_data_1_prepared["f0"] * geo_data_1_prepared["f2"]
geo_data_1_prepared = geo_data_1_prepared.drop(["f0", "f2"], axis=1)
geo_data_1_prepared.head()

##### geo_data_2

Для набора `geo_data_2` нет необходимости в дополнительных преобразованиях

In [ ]:
geo_data_2_prepared = geo_data_2.copy()
geo_data_2_prepared.head()

##### geo_data_full

Для набора `geo_data_full` нет необходимости в дополнительных преобразованиях

In [ ]:
geo_data_full_prepared = geo_data_full.copy()
geo_data_full_prepared.head()

Построим тепловую карту для подготовленных данных

In [ ]:
df_dict_prepared = {
    "geo_data_0_prepared": geo_data_0_prepared,
    "geo_data_1_prepared": geo_data_1_prepared,
    "geo_data_2_prepared": geo_data_2_prepared,
    "geo_data_full_prepared": geo_data_full_prepared,
}

# Создаем фигуру с тремя подграфиками
plt.figure(figsize=(15, 3))

for index, key in enumerate(df_dict_prepared):
    # подграфик для каждого датасета
    plt.subplot(1, len(df_dict_prepared), index + 1)
    sns.heatmap(df_dict_prepared[key].phik_matrix(verbose=False), annot=True, fmt=".2f")
    plt.title(key)

plt.tight_layout()
plt.show()

Мультиколлинеарности и сильной корреляции между нецелевыми признаками больше нет

#### Создание наборов для обучения моделей

Подготовим тестовые данные, разобъем на выборки по признакам

In [ ]:
RANDOM_STATE = 1
TEST_SIZE = 0.25

X_train_geo_0, X_test_geo_0, y_train_geo_0, y_test_geo_0 = train_test_split(
    geo_data_0.drop(["product"], axis=1),
    geo_data_0["product"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train_geo_1, X_test_geo_1, y_train_geo_1, y_test_geo_1 = train_test_split(
    geo_data_1.drop(["product"], axis=1),
    geo_data_1["product"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train_geo_2, X_test_geo_2, y_train_geo_2, y_test_geo_2 = train_test_split(
    geo_data_2.drop(["product"], axis=1),
    geo_data_2["product"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train_geo_full, X_test_geo_full, y_train_geo_full, y_test_geo_full = train_test_split(
    geo_data_full.drop(["product"], axis=1),
    geo_data_full["product"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train_geo_0.shape, X_test_geo_0.shape, X_train_geo_1.shape, X_test_geo_1.shape, X_train_geo_2.shape, X_test_geo_2.shape, X_train_geo_full.shape, X_test_geo_full.shape

### Промежуточные выводы

Проведен анализ и подготовка данных

- выявлены дубликаты в поле `id`
- проведен анализ распределений признаков. Некоторые признаки имеют отличное от нормального распределение
- проведен анализ корреляции признаков между собой. Мультиколлинеарность и сильная корреляция между нецелевыми признаками была устранена с помощью создания новых признаков
- данные подготовлены для обучения и проверки моделей

## Обучение и проверка модели

Подготовим функцию оценки RMSE

In [ ]:
# Создаем пользовательскую функцию оценки
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Создаем объект оценки
rmse_scorer = make_scorer(rmse, greater_is_better=False)

Подготовим функцию для создания пайплайна и обучения моделей

In [ ]:
def create_pipe_get_random_search(num_columns):

    # пайплайн для подготовки данных
    data_preprocessor = ColumnTransformer(
        [
            ("num", MinMaxScaler(), num_columns),
        ],
        remainder="passthrough",
    )

    # итоговый пайплайн: подготовка данных и модель
    pipe_final = Pipeline(
        [
            ("preprocessor", data_preprocessor),
            ("models", LinearRegression()),
        ]
    )

    # Подготовим словари для моделей данных
    param_grid = [
        # словарь для модели LinearRegression()
        {
            "models": [LinearRegression()],
            "models__fit_intercept": [True, False],
            # "models__normalize": [True, False],
            "preprocessor__num": [
                StandardScaler(),
                MinMaxScaler(),
                RobustScaler(),
                "passthrough",
            ],
        },
    ]

    grid_search = GridSearchCV(
        pipe_final,
        param_grid,
        cv=5,
        scoring=rmse_scorer,
        n_jobs=-1,
    )

    return grid_search

In [ ]:
# вывод результатов и получение лучшей модели
def get_model(X_train, y_train):
    num_columns = X_train.columns.to_list()

    # Обучение модели
    grid_search = create_pipe_get_random_search(num_columns)
    grid_search.fit(X_train, y_train)

    # Получение результатов
    print("Лучшие параметры:", grid_search.best_params_)
    print("Лучший RMSE:", -grid_search.best_score_)

    return grid_search.best_estimator_

Получим предсказания для объединенного датасета

In [ ]:
geo_data_full_best = get_model(X_train_geo_full, y_train_geo_full)

# print("-" * 50, f"\n\ngeo_data_0:\n")
# geo_data_0_best = get_model(X_train_geo_0, y_train_geo_0)
# print("-" * 50, f"\n\ngeo_data_1:\n")
# geo_data_1_best = get_model(X_train_geo_1, y_train_geo_1)
# print("-" * 50, f"\n\ngeo_data_2:\n")
# geo_data_2_best = get_model(X_train_geo_2, y_train_geo_2)

Выделим препроцессор и модель

In [ ]:
preprocessor = geo_data_full_best.named_steps['preprocessor']
model = geo_data_full_best.named_steps['models']

Получим предсказания для наборов данных по регионам

In [ ]:
# подготовим тестовую выборку
# preprocessor = geo_data_0_best.named_steps['preprocessor']
# model = geo_data_0_best.named_steps['models']
X_test_geo_0_fit = preprocessor.transform(X_test_geo_0.copy())

# предскажем значения
y_test_geo_0_pred = model.predict(request=X_test_geo_0_fit)
geo_data_0_pred = X_test_geo_0.copy()
geo_data_0_pred["product"] = y_test_geo_0
geo_data_0_pred["product_pred"] = y_test_geo_0_pred
display(geo_data_0_pred.head())

# средний запас предсказанного сырья и RMSE модели
geo_data_0_product_pred_avg = sum(y_test_geo_0_pred) / len(y_test_geo_0_pred)
print(f"Средний запас реального сырья по региону: {sum(y_test_geo_0) / len(y_test_geo_0)}")
print(f"Средний запас предсказанного сырья по региону: {geo_data_0_product_pred_avg}")
print(f"RMSE модели на тестовых данных: {rmse(y_test_geo_0, y_test_geo_0_pred)}")

In [ ]:
# подготовим тестовую выборку
# preprocessor = geo_data_1_best.named_steps['preprocessor']
# model = geo_data_1_best.named_steps['models']
X_test_geo_1_fit = preprocessor.transform(X_test_geo_1.copy())

# предскажем значения
y_test_geo_1_pred = model.predict(request=X_test_geo_1_fit)
geo_data_1_pred = X_test_geo_1.copy()
geo_data_1_pred["product"] = y_test_geo_1
geo_data_1_pred["product_pred"] = y_test_geo_1_pred
display(geo_data_1_pred.head())

# средний запас предсказанного сырья и RMSE модели
geo_data_1_product_pred_avg = sum(y_test_geo_1_pred) / len(y_test_geo_1_pred)
print(f"Средний запас реального сырья по региону: {sum(y_test_geo_1) / len(y_test_geo_1)}")
print(f"Средний запас предсказанного сырья по региону: {geo_data_1_product_pred_avg}")
print(f"RMSE модели на тестовых данных: {rmse(y_test_geo_1, y_test_geo_1_pred)}")

In [ ]:
# подготовим тестовую выборку
# preprocessor = geo_data_2_best.named_steps['preprocessor']
# model = geo_data_2_best.named_steps['models']
X_test_geo_2_fit = preprocessor.transform(X_test_geo_2.copy())

# предскажем значения
y_test_geo_2_pred = model.predict(request=X_test_geo_2_fit)
geo_data_2_pred = X_test_geo_2.copy()
geo_data_2_pred["product"] = y_test_geo_2
geo_data_2_pred["product_pred"] = y_test_geo_2_pred
display(geo_data_2_pred.head())

# средний запас предсказанного сырья и RMSE модели
geo_data_2_product_pred_avg = sum(y_test_geo_2_pred) / len(y_test_geo_2_pred)
print(f"Средний запас реального сырья по региону: {sum(y_test_geo_2) / len(y_test_geo_2)}")
print(f"Средний запас предсказанного сырья по региону: {geo_data_2_product_pred_avg}")
print(f"RMSE модели на тестовых данных: {rmse(y_test_geo_2, y_test_geo_2_pred)}")

### Промежуточные выводы

На тестовых данных модели показали результаты, близкие к результатам на тренировочных данных.

## Подготовка к расчёту прибыли

In [ ]:
BUDGET = 10_000_000_000
REVENUE_PER_UNIT = 450_000
LOSS_PROBABILITY_THRESHOLD = 0.025

sufficient_volume_of_raw_materials = BUDGET / REVENUE_PER_UNIT
print(f"Необходимый общий объем сырья для безубыточной разработки: {round(sufficient_volume_of_raw_materials, 2)} тысяч баррелей")

Найдем минимальное необходимое количество скважин для региона исходя из среднего значения предсказанного сырья.

In [ ]:
print("Минимальное необходимое количество скважин для региона geo_data_0:", round(sufficient_volume_of_raw_materials / geo_data_0_product_pred_avg, 2))
print("Минимальное необходимое количество скважин для региона geo_data_1:", round(sufficient_volume_of_raw_materials / geo_data_1_product_pred_avg, 2))
print("Минимальное необходимое количество скважин для региона geo_data_2:", round(sufficient_volume_of_raw_materials / geo_data_2_product_pred_avg, 2))

Судя по среднему значению, запасов сырья недостаточно для безубыточной разработки во всех регионах, т.к. бюджет выделен только для 200 скважин

Отсортируем данные по регионам по убыванию предсказанных запасов и выберем первых 200. Найдем объем запасов для них и сравним с минимально необходимым

In [ ]:
# функция для подсчета прибыли
def total_revenue(budget, revenue_per_unit, df, df_product_column):
    return (df[df_product_column].sum() - budget / revenue_per_unit) * revenue_per_unit

In [ ]:
geo_data_0_pred_largest = geo_data_0_pred.nlargest(200, "product_pred")
print(f"Total revenue, geo_data_0: {total_revenue(BUDGET, REVENUE_PER_UNIT, geo_data_0_pred_largest, 'product_pred')}")

geo_data_1_pred_largest = geo_data_1_pred.nlargest(200, "product_pred")
print(f"Total revenue, geo_data_1: {total_revenue(BUDGET, REVENUE_PER_UNIT, geo_data_1_pred_largest, 'product_pred')}")

geo_data_2_pred_largest = geo_data_2_pred.nlargest(200, "product_pred")
print(f"Total revenue, geo_data_2: {total_revenue(BUDGET, REVENUE_PER_UNIT, geo_data_2_pred_largest, 'product_pred')}")

### Промежуточные выводы

Регион `geo_data_1` выглядит наименее привлекательно по количеству потенциальной прибыли.

## Расчёт прибыли и рисков 

Для расчета прибыли и рисков применим технику Bootstrap для каждого региона

In [ ]:
# функция вывода гистрограммы распределения прибыли со средним значением и границами доверительного интервала
def revenue_hist(df, mean, confidence_interval):
    # Создаем гистограмму
    fig = px.histogram(
        df,
        nbins=100,
        title='Распределение значений прибыли',
        opacity=0.7
    )

    # Добавляем вертикальные линии
    # Линия для среднего значения
    fig.add_shape(
        type="line",
        x0=mean,  # координата x
        y0=0,  # начало линии (относительные координаты)
        x1=mean,  # конец линии по x
        y1=45,  # конец линии (относительные координаты)
        line=dict(
            color="blue",
            width=2,
            dash="dash"  # пунктирная линия
        )
    )

    # Линия для доверительного интервала (нижний порог)
    fig.add_shape(
        type="line",
        x0=confidence_interval[0],
        y0=0,
        x1=confidence_interval[0],
        y1=45,
        line=dict(
            color="red",
            width=2,
            dash="dot"
        )
    )

    # Линия для доверительного интервала (верхний порог)
    fig.add_shape(
        type="line",
        x0=confidence_interval[1],
        y0=0,
        x1=confidence_interval[1],
        y1=45,
        line=dict(
            color="red",
            width=2,
            dash="dot"
        )
    )

    # Добавляем аннотации к линиям
    fig.add_annotation(
        x=mean,
        y=-2,  # относительные координаты
        text="Средняя прибыль",
        showarrow=False,
        font=dict(size=12, color="blue")
    )

    fig.add_annotation(
        x=confidence_interval[0],
        y=-2,
        text="Доверительный интервал",
        showarrow=False,
        font=dict(size=12, color="red")
    )

    fig.add_annotation(
        x=confidence_interval[1],
        y=-2,
        text="Доверительный интервал",
        showarrow=False,
        font=dict(size=12, color="red")
    )

    # Настраиваем внешний вид графика
    fig.update_layout(
        xaxis_title='Прибыль',
        yaxis_title='Частота',
        showlegend=False,
        plot_bgcolor='white'
    )

    fig.show()


In [ ]:
STATE = np.random.RandomState(1)
BOOTSTRAP_SAMPLES = 1000

def region_decision(df):
    revenue_values = []
    # Создание 1000 случайных выборок
    for i in range(BOOTSTRAP_SAMPLES):
        # Случайным образом выбираем 500 скважин
        target_subsample = df.sample(n=500, replace=True, random_state=STATE)
        # Сортируем по прогнозируемому запасу и берем топ200
        target_subsample_largest = target_subsample.nlargest(200, "product_pred")

        # По этим точкам берем фактические значения запасов и считаем прибыль для каждой выборки
        revenue_values.append(total_revenue(BUDGET, REVENUE_PER_UNIT, target_subsample_largest, "product"))

    revenue_values = pd.Series(revenue_values)
    df = revenue_values.count() - 1

    # Расчет доверительного интервала
    confidence_interval = st.t.interval(0.95, df, loc=revenue_values.mean(), scale=revenue_values.sem())
        
    # Вывод результатов
    print("Средняя прибыль:", revenue_values.mean())
    print("95% доверительный интервал (истинное среднее):", confidence_interval)
    confidence_interval_pred = (revenue_values.quantile(0.025), revenue_values.quantile(0.975))
    print("95% доверительный интервал (прогноз):", confidence_interval_pred)

    # Расчет вероятности убытка
    loss_probability = revenue_values[revenue_values < 0].count() / revenue_values.count()
    decision = "оставляем" if loss_probability < LOSS_PROBABILITY_THRESHOLD else "не оставляем"
    print(f"Вероятность убытка = {loss_probability:.2%}, решение: {decision}\n")

    
    revenue_hist(revenue_values, revenue_values.mean(), confidence_interval_pred)


# выведем статистику для каждого региона
regions = [geo_data_0_pred, geo_data_1_pred, geo_data_2_pred]
for region_index, df in enumerate(regions):
    print("-" * 50)
    print(f"Регион: {region_index}\n")
    region_decision(df)
    print()

### Промежуточные выводы

Ни один регион не подходит для разработки, т.к. во всех превышен порог вероятности убытка

## Общие выводы

В проекте было проведено исследование по выбору региона для разработки месторождений нефтяных скважин. 

В ходе проекта были выполнены следующие шаги:
- загрузка и подготовка данных
- первичный анализ данных
- исследование на наличие дубликатов в данных
- исследовательский анализ данных
- корреляционный анализ признаков
- созданы новые признаки
- проведено обучение и проверка модели
- проведен расчет прибыли и рисков

По результатам проведенной работы ни один регион нельзя рекомендовать к разработке месторождений, т.к. во всех превышен порог вероятности убытка.

## Чек-лист готовности проекта

Поставьте 'x' в выполненных пунктах. Далее нажмите Shift+Enter.

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Выполнен шаг 1: данные подготовлены
- [x]  Выполнен шаг 2: модели обучены и проверены
    - [x]  Данные корректно разбиты на обучающую и валидационную выборки
    - [x]  Модели обучены, предсказания сделаны
    - [x]  Предсказания и правильные ответы на валидационной выборке сохранены
    - [x]  На экране напечатаны результаты
    - [x]  Сделаны выводы
- [x]  Выполнен шаг 3: проведена подготовка к расчёту прибыли
    - [x]  Для всех ключевых значений созданы константы Python
    - [x]  Посчитано минимальное среднее количество продукта в месторождениях региона, достаточное для разработки
    - [x]  По предыдущему пункту сделаны выводы
    - [x]  Написана функция расчёта прибыли
- [x]  Выполнен шаг 4: посчитаны риски и прибыль
    - [x]  Проведена процедура *Bootstrap*
    - [x]  Все параметры бутстрепа соответствуют условию
    - [x]  Найдены все нужные величины
    - [x]  Предложен регион для разработки месторождения
    - [x]  Выбор региона обоснован